In [ ]:
from glob import glob

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
class LogsParser():
    _last_field = ""
    expected_text_by_field = {}

    def __init__(self): 
        self.data = []

        self._data_dict = {}
        self._field_to_get = ""
        self._expected_text = ""
        
        self._update_params()

        self._update_function_by_field = {}

    def _set_field_to_get(self):
        raise NotImplementedError("Subclass must implement this method")

    def _update_data_dict(self, line):
        update_function = self._update_function_by_field.get(self._field_to_get, None)
        if update_function is None:
            return
        
        update_function(line)
    
    def _update_params(self):
        if self._field_to_get == self._last_field:
            self.data.append(self._data_dict)
            self._data_dict = {}

        self._set_field_to_get()
        self._expected_text = self.expected_text_by_field[self._field_to_get]
    
    def parse_logs(self, logs_folder):
        logs_paths = sorted(glob(f"{logs_folder}/encryption.tiempos*.log"))

        self.data = []

        for log_path in logs_paths:
            with open(log_path, "r") as file:
                for line in file:
                    if self._expected_text not in line:
                        continue

                    self._update_data_dict(line)
                    self._update_params()

        return self.data


In [ ]:
class MessagesLogsParser(LogsParser):
    expected_text_by_field = {
        "n_characters": "letters",
        "n_words": "Encryption of a message",
        "decryption_time": "Decryption of a message"
    }
    _last_field = "decryption_time"

    def __init__(self): 
        super().__init__()

        self._update_function_by_field = {
            "n_characters": self._extract_n_characters_data,
            "n_words": self._extract_n_words_data,
            "decryption_time": self._extract_decryption_time_data
        }

    def _set_field_to_get(self):
        if self._data_dict.get("n_characters") is None:
            self._field_to_get = "n_characters"
            return
        
        if self._data_dict.get("n_words") is None:
            self._field_to_get = "n_words"
            return
        
        if self._data_dict.get("decryption_time") is None:
            self._field_to_get = "decryption_time"
            return

    def _extract_n_characters_data(self, line):
        n_characters = line.split(" letters ")[0].split(" ")[-1]
        self._data_dict["n_characters"] = int(n_characters)

    def _extract_n_words_data(self, line):
        n_words = line.split(" words ")[0].split(" ")[-1]
        self._data_dict["n_words"] = int(n_words)

        encryption_time = line.split(" ms\n")[0].split(" ")[-1]
        self._data_dict["encryption_time"] = int(encryption_time)

    def _extract_decryption_time_data(self, line):
        decryption_time = line.split(" ms\n")[0].split(" ")[-1]
        self._data_dict["decryption_time"] = int(decryption_time)

In [ ]:
class WordsLogsParser(LogsParser):
    expected_text_by_field = {
        "encryption": "Encryption of characters",
        "encryption_swap": "Swapping",
        "decryption": "Decryption of characters",
        "decryption_swap": "Swapping",
    }
    _last_field = "message"

    def __init__(self): 
        self.data = []

        self._data_dict = {}
        self._field_to_get = "swap" # so it changes to encryption
        self._expected_text = ""
        self._current_mode = "encryption"
        
        self._update_params()

    def _set_field_to_get(self):
        if "swap" in self._field_to_get:
            self._field_to_get = f"{self._current_mode}"
            return
        
        self._field_to_get = f"{self._current_mode}_swap"

    def _update_data_dict(self, line):
        n_characters = line.split(" letters ")[0].split(" ")[-1]
        self._data_dict["n_characters"] = int(n_characters)

        time = line.split(" ns\n")[0].split(" ")[-1]
        self._data_dict["time"] = int(time)
        
        self._data_dict["task"] = self._field_to_get

    def _swap_mode(self):
        self._current_mode = "decryption" if self._current_mode == "encryption" else "encryption"
    
    def _update_params(self):
        if len(self._data_dict) != 0:
            self.data.append(self._data_dict)
        self._data_dict = {}
        
        self._set_field_to_get()
        self._expected_text = self.expected_text_by_field[self._field_to_get]
    
    def parse_logs(self, logs_folder):
        logs_paths = sorted(glob(f"{logs_folder}/encryption.tiempos*.log"))

        self.data = []

        for log_path in logs_paths:
            with open(log_path, "r") as file:
                for line in file:
                    if "message" in line:
                        self._swap_mode()
                        self._update_params()
                        continue
                    
                    if self._expected_text not in line:
                        continue

                    self._update_data_dict(line)
                    self._update_params()

        return self.data


In [ ]:
def plot_data(df, ax, field, hue=None):
    ax.set_xlabel("Número de palabras")
    
    if hue is not None:
        sns.lineplot(
            x="n_words", y=field, data=df, ax=ax, hue=hue
        )
        return
    
    sns.lineplot(
        x="n_words", y=field, data=df, ax=ax, color="black"
    )
    
    sns.scatterplot(
        x="n_words", y=field, data=df, color="black", ax=ax
    )

def plot_polyfit(df, ax, field, polyfit_order):
    x = df["n_words"]
    y = df[field]

    slope, intercept = np.polyfit(x**polyfit_order, y, 1)
    x = np.linspace(0, 1500, 50)

    sns.lineplot(
        x=x, y=slope*x**polyfit_order + intercept, color="gray", marker=".",
        linewidth=0.5, ax=ax
    )
    ax.text(0, 0.95*max(y), f"$y = {slope:.2e}x^{polyfit_order} + {intercept:.2e}$")

In [ ]:
logs_parser = MessagesLogsParser()
average_case_data = logs_parser.parse_logs("logs_data/average_case")
worst_case_data = logs_parser.parse_logs("logs_data/worst_case")

df_w = pd.DataFrame(worst_case_data)
df_a = pd.DataFrame(average_case_data)
df_af = df_a.loc[df_a["n_characters"] <= 50].copy().reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 4))

plot_polyfit(df_w, ax[0], "encryption_time", 3)
plot_data(df_w, ax[0], "encryption_time")
ax[0].set_ylabel("Tiempo de encriptación (ms)")
ax[0].set_title("Peor escenario (n^3)")

plot_data(df_af, ax[1], "encryption_time", hue="n_characters")
ax[1].set_ylabel("Tiempo de encriptación (ms)")
ax[1].set_title("Escenario promedio O(m*n)~O(n)")


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 4))

plot_polyfit(df_w, ax[0], "decryption_time", 3)
plot_data(df_w, ax[0], "decryption_time")
ax[0].set_ylabel("Tiempo de desencriptación (ms)")
ax[0].set_title("Peor escenario (n^3)")

plot_data(df_af, ax[1], "decryption_time", hue="n_characters")
ax[1].set_ylabel("Tiempo de desencriptación (ms)")
ax[1].set_title("Escenario promedio O(m*n)~O(n)")


In [ ]:
logs_parser = WordsLogsParser()
words_data = logs_parser.parse_logs("logs_data/worst_case")

df_words = pd.DataFrame(words_data)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5, 3.5))

df_words_f = df_words.loc[df_words["task"].str.contains("encryption")].copy().reset_index(drop=True)

sns.lineplot(
    x="n_characters", y="time", data=df_words_f, ax=ax, hue="task"
)
ax.set_xlabel("Número de caracteres")
ax.set_ylabel("Tiempo de encriptación (ns)")
ax.set_ylim(-100, 10100)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5, 3.5))

df_words_f = df_words.loc[df_words["task"].str.contains("decryption")].copy().reset_index(drop=True)
df_words_f.sort_values(by="task", inplace=True)

sns.lineplot(
    x="n_characters", y="time", data=df_words_f, ax=ax, hue="task"
)
ax.set_xlabel("Número de caracteres")
ax.set_ylabel("Tiempo de desencriptación (ns)")
ax.set_ylim(-100, 10100)